<a href="https://colab.research.google.com/github/zixian0821-zoe/stochastic_search_and_optimization/blob/main/hw8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import numpy as np

np.random.seed(42)


fit_path = '/content/drive/MyDrive/reeddata-fit-randomized.xlsx'
test_path = '/content/drive/MyDrive/reeddata-test.xls'

df_fit = pd.read_excel(
    fit_path,
    sheet_name='raw and curvilinear data',
    header=None
)

df_test = pd.read_excel(
    test_path,
    header=None
)

z_train = pd.to_numeric(df_fit.iloc[2:162, 0], errors='coerce').values
X_train = df_fit.iloc[2:162, 1:7].apply(pd.to_numeric, errors='coerce').values

z_test = pd.to_numeric(df_test.iloc[2:82, 0], errors='coerce').values
X_test = df_test.iloc[2:82, 1:7].apply(pd.to_numeric, errors='coerce').values

p = 13

def h_predict(theta, x):
    return theta[0] + np.dot(theta[1:7], x) + np.dot(theta[7:13], x**2)

def dh_dtheta(theta, x):
    grad = np.zeros(p)
    grad[0] = 1.0
    grad[1:7] = x
    grad[7:13] = x**2
    return grad

def run_recursive_SA(X, z, a_val, theta0):
    n = len(z)
    theta = theta0.copy()
    for k in range(n):
        a_k = a_val / (k + 10)**0.501
        residual = h_predict(theta, X[k]) - z[k]
        Y_k = residual * dh_dtheta(theta, X[k])
        theta = theta - a_k * Y_k
    return theta

def compute_MAD(theta, X, z):
    preds = np.array([h_predict(theta, X[i]) for i in range(len(z))])
    return np.mean(np.abs(z - preds))

theta0 = np.zeros(p)
theta0[7:13] = 1.0

for a in [0.005, 0.02, 0.05]:
    theta_est = run_recursive_SA(X_train, z_train, a, theta0)
    print(f"a={a}: MAD_train={compute_MAD(theta_est, X_train, z_train):.6f}, "
          f"MAD_test={compute_MAD(theta_est, X_test, z_test):.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
a=0.005: MAD_train=0.735207, MAD_test=0.548576
a=0.02: MAD_train=0.564181, MAD_test=0.445886
a=0.05: MAD_train=0.452154, MAD_test=0.337894
